In [1]:
%pip install -q transformers torch tqdm


[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.4.0_1/libexec/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification
import warnings
warnings.filterwarnings("ignore")

print("Loading model... (first time downloads ~250MB)")
tokenizer = AutoTokenizer.from_pretrained("d4data/biomedical-ner-all")
model = AutoModelForTokenClassification.from_pretrained("d4data/biomedical-ner-all")

ner_pipe = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
)

# Sanity check, does the model actually work?
test_result = ner_pipe("A 58-year-old woman with hypertension and chest pain was prescribed aspirin.")
for ent in test_result:
    print(f"  {ent['entity_group']:25s} | {ent['word']:30s} | {ent['score']:.3f}")

print("\n✓ Model loaded successfully")

/opt/homebrew/Cellar/jupyterlab/4.4.0_1/libexec/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model... (first time downloads ~250MB)


Loading weights: 100%|█| 102/102 [00:00<00:00, 1964.11it/s, Materializing param=


  Age                       | 58 - year - old                | 0.999
  Sex                       | woman                          | 1.000
  Sign_symptom              | hypertension                   | 0.980
  Biological_structure      | chest                          | 1.000
  Sign_symptom              | pain                           | 1.000

✓ Model loaded successfully


In [3]:
%pip install pandas tqdm


[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.4.0_1/libexec/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import re
from tqdm import tqdm

chunks_df = pd.read_csv("open_patients_chunks.csv")
chunks_df["text"] = chunks_df["text"].fillna("").astype(str)
chunks_df = chunks_df[chunks_df["text"].str.strip().ne("")].copy()
print(f"Loaded {len(chunks_df)} chunks from {chunks_df['_id'].nunique()} records")

# We only care about these 4 entity types, skip everything else
TARGET_TYPES = {
    "Disease_disorder":      "Condition",
    "Sign_symptom":          "Symptom",
    "Medication":            "Medication",
    "Therapeutic_procedure": "Procedure",
    "Diagnostic_procedure":  "Procedure",
}

VALID_SHORT = {"ct", "mri", "ecg", "ekg", "icu", "iv", "copd", "hiv", "uti", "mi", "cva", "cabg"}

def is_word_char(c):
    return c.isalnum() or c in ["_", "-", "/"]

def span_is_full_word(text, start, end):
    left_ok = (start == 0) or (not is_word_char(text[start - 1]))
    right_ok = (end >= len(text)) or (not is_word_char(text[end]))
    return left_ok and right_ok

def basic_span_clean(x):
    if not isinstance(x, str):
        return None
    x = x.strip()
    x = re.sub(r"\s+", " ", x)
    x = x.strip(' .,;:()[]{}"\'')
    x = x.lower()
    if not x:
        return None
    if "##" in x:
        return None
    if not re.search(r"[a-z]", x):
        return None
    if len(x) <= 3 and x not in VALID_SHORT:
        return None
    return x

# Run NER across all chunks
all_entities = []
errors = []

for idx, row in tqdm(chunks_df.iterrows(), total=len(chunks_df), desc="NER extraction"):
    try:
        text = row["text"]
        results = ner_pipe(text)

        for ent in results:
            # Not one of our 4 target types? Skip it
            entity_type = ent["entity_group"]
            if entity_type not in TARGET_TYPES:
                continue

            start = int(ent["start"])
            end   = int(ent["end"])
            score = float(ent["score"])

            surface = text[start:end]

            if not span_is_full_word(text, start, end):
                continue

            surface_clean = basic_span_clean(surface)
            if surface_clean is None:
                continue

            all_entities.append({
                "chunk_id":         row["chunk_id"],
                "_id":              row["_id"],
                "source":           row["source"],
                "chunk_index":      row["chunk_index"],
                "entity_text":      surface,
                "entity_clean":     surface_clean,
                "entity_type":      entity_type,
                "category":         TARGET_TYPES[entity_type],  # map to our category right away
                "confidence":       round(score, 4),
                "start":            start,
                "end":              end
            })

    except Exception as e:
        errors.append({"chunk_id": row.get("chunk_id", idx), "error": str(e)})

raw_df = pd.DataFrame(all_entities)

print(f"\n{'='*60}")
print(f"Raw entities extracted : {len(raw_df)}")
print(f"Chunks with errors     : {len(errors)}")

if not raw_df.empty:
    print(f"\nCategory distribution:")
    print(raw_df["category"].value_counts().to_string())

if errors:
    pd.DataFrame(errors).to_csv("ner_errors.csv", index=False)
    print("\nSaved errors to ner_errors.csv")

Loaded 1667 chunks from 1000 records


NER extraction: 100%|███████████████████████| 1667/1667 [00:53<00:00, 31.24it/s]


Raw entities extracted : 21361
Chunks with errors     : 0

Category distribution:
category
Procedure     11142
Symptom        7463
Medication     1387
Condition      1369


In [5]:
import os
raw_df.to_csv("raw_entities.csv", index=False)
print(f"Saved {len(raw_df)} raw entities to raw_entities.csv")
print(f"File size: {os.path.getsize('raw_entities.csv') / 1024:.1f} KB")

Saved 21361 raw entities to raw_entities.csv
File size: 2244.3 KB


In [ ]:
filtered_df = raw_df.copy()
# Short abbreviations that are legit medical terms, don't filter these out
VALID_SHORT = {
    "ct", "mri", "ecg", "ekg", "icu", "iv", "hiv", "uti",
    "mi", "cva", "dvt", "tia", "sle", "osa", "bph", "ckd",
    "asa", "chf", "cad", "pci",
}

# These words are too vague to be useful as standalone entities
GENERIC_DROP = {
    "medication", "medications", "drug", "drugs",
    "symptom", "symptoms", "condition", "conditions",
    "procedure", "procedures", "test", "tests",
    "exam", "examination", "treatment", "therapy",
    "disease", "disorder", "syndrome",
}

# The model sometimes picks up filler words, drop them
SKIP_TERMS = {
    "patient", "patients", "she", "her", "his", "him", "he",
    "the", "was", "were", "are", "been", "normal", "right",
    "left", "well", "mild", "moderate", "severe", "history",
    "noted", "revealed", "showed", "presented", "performed",
    "including", "consistent", "complains", "complaint",
    "general", "acute", "chronic", "bilateral", "unilateral",
    "positive", "negative", "stable", "above", "below",
    "initial", "subsequent", "follow-up", "overall",
    "recovery", "diagnosis", "management", "evaluation",
}

# Noise that only shows up in certain categories
CATEGORY_NOISE = {
    "Condition": {"pulmonary", "acute", "breast", "renal", "hepatic"},
    "Symptom":   {"enlarged", "elevated", "decreased", "increased",
                  "reduced", "abnormal", "irregular", "diffuse",
                  "positive", "negative"},
    "Medication": {"general", "therapy", "supplement",
                   "supplementation", "iv", "general anesthesia",
                   "chemotherapy"},  # treatment modalities, not specific drugs
    "Procedure":  {"blood", "serum", "plasma", "specimen",
                   "sample", "fluid", "tissue"},
}

print(f"Config loaded: {len(VALID_SHORT)} valid abbreviations, "
      f"{len(GENERIC_DROP)} generic drops, {len(SKIP_TERMS)} skip terms")

Config loaded: 20 valid abbreviations, 19 generic drops, 47 skip terms


In [7]:
VARIANT_MERGE = {
    # --- PROCEDURES ---
    "computed tomography":       "ct",
    "ct scan":                   "ct",
    "ct scanning":               "ct",
    "magnetic resonance imaging":"mri",
    "mri scan":                  "mri",
    "mri scanning":              "mri",
    "electrocardiogram":         "ecg",
    "ekg":                       "ecg",
    "electrocardiography":       "ecg",
    "electroencephalogram":      "eeg",
    "electroencephalography":    "eeg",
    "chest x-ray":               "x-ray",
    "chest radiograph":          "x-ray",
    "radiograph":                "x-ray",
    "ultrasonography":           "ultrasound",
    "echocardiogram":            "echocardiography",

    # Procedure name cleanup
    "vital signs":               "vital sign",
    "laboratory tests":          "laboratory test",
    "laboratory findings":       "laboratory test",
    "laboratory investigation":  "laboratory test",
    "laboratory investigations": "laboratory test",
    "blood tests":               "blood test",
    "blood cultures":            "blood culture",
    "physical exam":             "physical examination",

    # --- CONDITIONS ---
    "diabetes mellitus":                   "diabetes",
    "type 2 diabetes mellitus":            "type 2 diabetes",
    "type 1 diabetes mellitus":            "type 1 diabetes",
    "coronary artery disease (cad)":       "coronary artery disease",
    "congestive heart failure (chf)":      "heart failure",
    "congestive heart failure":            "heart failure",
    "hypertension (htn)":                  "hypertension",
    "chronic obstructive pulmonary disease (copd)": "copd",
    "chronic obstructive pulmonary disease":        "copd",
    "deep vein thrombosis (dvt)":          "deep vein thrombosis",
    "atrial fibrillation (afib)":          "atrial fibrillation",
    "cerebrovascular accident (cva)":      "stroke",
    "cerebrovascular accident":            "stroke",
    "cva":                                 "stroke",
    "urinary tract infection (uti)":       "urinary tract infection",
    "hiv infection":                       "hiv",

    # --- SYMPTOMS ---
    "chest pains":     "chest pain",
    "abdominal pains": "abdominal pain",
    "headaches":       "headache",
    "seizures":        "seizure",
    "metastases":      "metastasis",   # FIX: was incorrectly "metastase"
    "tumors":          "tumor",
    "tumour":          "tumor",
    "tumours":         "tumor",
    "lesions":         "lesion",
    "nodules":         "nodule",
    "masses":          "mass",
    "fractures":       "fracture",
    "infections":      "infection",
    "complications":   "complication",
    "abnormalities":   "abnormality",
    "shortness of breath (sob)": "shortness of breath",

    # --- MEDICATIONS ---
    "antibiotics":    "antibiotic",
    "steroids":       "steroid",
    "corticosteroids":"corticosteroid",
    "nsaids":         "nsaid",
    "nonsteroidal anti-inflammatory drugs (nsaids)": "nsaid",
    "aspirin (asa)":  "aspirin",
}

print(f"Variant merge rules: {len(VARIANT_MERGE)}")

Variant merge rules: 62


In [ ]:
# Medical terms that end in s/is/us but aren't plurals, leave them alone
NO_SINGULARIZE = {
    "diabetes", "herpes", "measles", "mumps", "rabies", "shingles",
    "arthritis", "sepsis", "psoriasis", "hepatitis", "thrombosis",
    "tuberculosis", "mellitus", "weakness", "tenderness", "abscess",
    "metastasis", "diagnosis", "prognosis", "stenosis", "fibrosis",
    "cirrhosis", "necrosis", "sclerosis", "acidosis", "alkalosis",
    "cyanosis", "scoliosis", "lordosis", "kyphosis", "ptosis",
    "mass", "loss", "consciousness", "stress", "illness",
    "virus", "fungus", "mucus", "thymus", "plexus",
    "species", "series",
}

PROTECTED_SUFFIXES = ("itis", "osis", "sis", "ness", "us", "ss", "is", "ias")

KNOWN_CONDITIONS = {
    "diabetes", "hypertension", "pneumonia", "infection", "fracture",
    "carcinoma", "adenocarcinoma", "thrombosis", "hepatitis", "hiv",
    "copd", "heart failure", "atrial fibrillation", "tuberculosis",
    "sepsis", "anemia", "asthma", "stroke", "cancer",
}

def clean_entity(text):
    if not isinstance(text, str):
        return None
    x = text.strip()
    x = re.sub(r"\s+", " ", x)
    x = x.strip(' .,;:()[]{}"\'')
    x = x.lower()
    if not x: return None
    if "##" in x: return None
    if not re.search(r"[a-z]", x): return None
    if x in GENERIC_DROP or x in SKIP_TERMS: return None
    if len(x) <= 3 and x not in VALID_SHORT: return None
    return x


def safe_singularize(token):
    t = token.lower()
    if t in NO_SINGULARIZE: return t
    if any(t.endswith(sfx) for sfx in PROTECTED_SUFFIXES): return t
    if len(t) <= 3: return t
    if t.endswith("ies") and len(t) > 4: return t[:-3] + "y"
    if t.endswith("sses") and len(t) > 5: return t[:-2]
    if t.endswith("ches") or t.endswith("shes") or t.endswith("xes") or t.endswith("zes"):
        return t[:-2]
    if t.endswith("s") and not t.endswith(("us", "is", "ss")):
        return t[:-1]
    return t


def singularize_last_word(text):
    if not text or not text.strip(): return text
    tokens = text.strip().split()
    tokens[-1] = safe_singularize(tokens[-1])
    return " ".join(tokens)


def normalize_entity(text, category=None):
    if text is None: return None
    x = str(text).strip().lower()
    x = re.sub(r"\s+", " ", x)
    x = x.strip(' .,;:()[]{}"\'\u2018\u2019\u201c\u201d`')
    if not x: return None
    if x in GENERIC_DROP or x in SKIP_TERMS: return None
    if category and category in CATEGORY_NOISE and x in CATEGORY_NOISE[category]:
        return None
    # Catch misclassifications — a condition name shouldn't be tagged as a medication
    if category == "Medication" and x in KNOWN_CONDITIONS:
        return None
    x = VARIANT_MERGE.get(x, x)
    x = singularize_last_word(x)
    x = VARIANT_MERGE.get(x, x)
    if x in GENERIC_DROP or x in SKIP_TERMS: return None
    # Check again after normalization since variant merge might have changed things
    if category == "Medication" and x in KNOWN_CONDITIONS:
        return None
    return x if x else None

print("Functions defined ✓")

Functions defined ✓


In [9]:
CONFIDENCE_THRESHOLD = 0.60  # Document this choice

before = len(filtered_df)
filtered_df = filtered_df[filtered_df['confidence'] >= CONFIDENCE_THRESHOLD].copy()
after = len(filtered_df)

print(f"Confidence filter (≥ {CONFIDENCE_THRESHOLD}): {before} → {after} entities")
print(f"Removed: {before - after} low-confidence entities")
print(f"\nRemaining by category:")
print(filtered_df['category'].value_counts().to_string())

Confidence filter (≥ 0.6): 21361 → 20515 entities
Removed: 846 low-confidence entities

Remaining by category:
category
Procedure     10719
Symptom        7163
Medication     1349
Condition      1284


In [10]:
source_col = "entity_clean" if "entity_clean" in filtered_df.columns else "entity_text"
filtered_df["entity_clean"] = filtered_df[source_col].apply(clean_entity)

before = len(filtered_df)
filtered_df = filtered_df[filtered_df["entity_clean"].notna()].copy()
print(f"Cleaning: {before} -> {len(filtered_df)} ({before - len(filtered_df)} removed)")

filtered_df["entity_norm"] = filtered_df.apply(
    lambda r: normalize_entity(r["entity_clean"], r.get("category")), axis=1
)

before = len(filtered_df)
filtered_df = filtered_df[filtered_df["entity_norm"].notna()].copy()
print(f"Normalization: {before} -> {len(filtered_df)} ({before - len(filtered_df)} removed)")

Cleaning: 20515 -> 19948 (567 removed)
Normalization: 19948 -> 19524 (424 removed)


In [11]:
ENTITY_KEY = "entity_norm"

# First pass: deduplicate within the same chunk
before = len(filtered_df)
deduped = filtered_df.sort_values("confidence", ascending=False).drop_duplicates(
    subset=["chunk_id", "category", ENTITY_KEY], keep="first"
)
print(f"Level 1 (within-chunk):  {before:6d} -> {len(deduped):6d}  (removed {before - len(deduped)})")

# Second pass: deduplicate across chunks of the same record
before = len(deduped)
deduped = deduped.sort_values("confidence", ascending=False).drop_duplicates(
    subset=["_id", "category", ENTITY_KEY], keep="first"
)
print(f"Level 2 (within-record): {before:6d} -> {len(deduped):6d}  (removed {before - len(deduped)})")

print(f"\nFinal: {len(deduped)} entity-record pairs across {deduped['_id'].nunique()} records")
print(f"\nPer category:")
print(deduped["category"].value_counts().to_string())

Level 1 (within-chunk):   19524 ->  17037  (removed 2487)
Level 2 (within-record):  17037 ->  15934  (removed 1103)

Final: 15934 entity-record pairs across 993 records

Per category:
category
Procedure     8695
Symptom       5201
Condition     1090
Medication     948


In [12]:
freq = (
    deduped
    .groupby(["category", ENTITY_KEY])
    .agg(
        record_count=("_id", "nunique"),
        avg_confidence=("confidence", "mean"),
        sources=("source", lambda x: dict(x.value_counts())),
    )
    .reset_index()
    .rename(columns={ENTITY_KEY: "entity"})
    .sort_values(["category", "record_count", "avg_confidence"],
                 ascending=[True, False, False])
)

top10_all = []
for cat in ["Condition", "Symptom", "Medication", "Procedure"]:
    top = freq[freq["category"] == cat].head(10).copy()
    top["rank"] = range(1, len(top) + 1)
    top10_all.append(top)

    print(f"\n{'=' * 60}")
    print(f"  TOP 10 {cat.upper()}S (by unique record count)")
    print(f"{'=' * 60}")
    for _, row in top.iterrows():
        print(f"  {row['rank']:2d}. {row['entity']:35s}  "
              f"{row['record_count']:4d} records  "
              f"(conf: {row['avg_confidence']:.2f})")

top10_df = pd.concat(top10_all, ignore_index=True)
top10_df = top10_df[["category", "rank", "entity", "record_count", "avg_confidence", "sources"]]


  TOP 10 CONDITIONS (by unique record count)
   1. infection                              27 records  (conf: 0.94)
   2. diabetes                               24 records  (conf: 0.90)
   3. pneumonia                              23 records  (conf: 0.97)
   4. adenocarcinoma                         19 records  (conf: 0.97)
   5. carcinoma                              19 records  (conf: 0.93)
   6. fracture                               13 records  (conf: 0.90)
   7. thrombosis                              9 records  (conf: 0.95)
   8. coronary artery disease                 9 records  (conf: 0.95)
   9. hiv                                     9 records  (conf: 0.93)
  10. hepatitis                               9 records  (conf: 0.91)

  TOP 10 SYMPTOMS (by unique record count)
   1. pain                                  300 records  (conf: 1.00)
   2. mass                                  181 records  (conf: 1.00)
   3. tumor                                 126 records  (conf: 0.99)


In [13]:
print("=== PER-SOURCE ENTITY DISTRIBUTION ===\n")

source_stats = (
    deduped.groupby(["source", "category"])
    .agg(
        entity_mentions=("entity_norm", "count"),
        unique_entities=("entity_norm", "nunique"),
        unique_records=("_id", "nunique"),
    )
    .reset_index()
)
print(source_stats.to_string(index=False))

# What are the top entities broken down by source?
print("\n--- Top 3 entities per source x category ---")
for source in ["pmc", "usmle", "trec-ct", "trec-cds"]:
    print(f"\n  [{source}]")
    src_data = deduped[deduped["source"] == source]
    for cat in ["Condition", "Symptom", "Medication", "Procedure"]:
        cat_data = src_data[src_data["category"] == cat]
        top3 = (cat_data.groupby("entity_norm")["_id"].nunique()
                .sort_values(ascending=False).head(3))
        names = ", ".join(f"{e} ({c})" for e, c in top3.items())
        print(f"    {cat:12s}: {names}")

=== PER-SOURCE ENTITY DISTRIBUTION ===

  source   category  entity_mentions  unique_entities  unique_records
     pmc  Condition              946              589             436
     pmc Medication              878              494             341
     pmc  Procedure             7888             3233             709
     pmc    Symptom             4498             1863             705
trec-cds  Condition               38               29              29
trec-cds Medication               14               14              12
trec-cds  Procedure              196              119              65
trec-cds    Symptom              219              128              80
 trec-ct  Condition               95               83              47
 trec-ct Medication               46               41              30
 trec-ct  Procedure              383              211             110
 trec-ct    Symptom              370              214             113
   usmle  Condition               11              

In [14]:
print("=== VALIDATION CHECKS ===\n")

# Eyeball the top 10 for near-duplicates we might have missed
for cat in ["Condition", "Symptom", "Medication", "Procedure"]:
    top = freq[freq["category"] == cat].head(10)
    entities = top["entity"].tolist()
    for i, e1 in enumerate(entities):
        for j, e2 in enumerate(entities):
            if i != j and (e1 in e2 or e2 in e1) and len(e1) > 2 and len(e2) > 2:
                print(f"  WARNING: {cat}: possible duplicate: '{e1}' vs '{e2}'")

# How many records actually produced at least one entity?
total = deduped["_id"].nunique()
print(f"\nRecords with >=1 entity: {total} / 1000")
for cat in ["Condition", "Symptom", "Medication", "Procedure"]:
    n = deduped[deduped["category"] == cat]["_id"].nunique()
    print(f"  {cat:12s}: {n:4d} records ({n/10:.1f}%)")

# How many entities got changed by normalization?
changed = deduped[deduped.get("entity_clean", deduped["entity_norm"]) != deduped["entity_norm"]]
print(f"\nEntities modified by normalization: {len(changed)}")
print("\n✓ Validation complete")

=== VALIDATION CHECKS ===


Records with >=1 entity: 993 / 1000
  Condition   :  522 records (52.2%)
  Symptom     :  943 records (94.3%)
  Medication  :  392 records (39.2%)
  Procedure   :  927 records (92.7%)

Entities modified by normalization: 2377

✓ Validation complete


In [ ]:
import random

# Pick 5 random records and show what we extracted, spot-check time
sample_ids = random.sample(list(deduped['_id'].unique()), 5)

for record_id in sample_ids:
    # Grab the source text
    original = chunks_df[chunks_df['_id'] == record_id]['text'].iloc[0][:300]
    # What did we pull out of this record?
    entities = deduped[deduped['_id'] == record_id][['category','entity_norm','confidence']]
    
    print(f"\n{'='*60}")
    print(f"Record: {record_id}")
    print(f"Text: {original}...")
    print(f"\nExtracted:")
    print(entities.to_string(index=False))


Record: pmc-5711658-1
Text: A 73-year-old woman with underlying diabetes and hypertension, but without a psychiatric history, complained of insomnia associated with visual and auditory hallucinations prior to sleeping, all of which persisted for one month. She described the hallucinations as hearing a voice talking about 'kill...

Extracted:
 category                   entity_norm  confidence
Procedure          cognitive assessment      0.9999
Procedure mini mental state examination      0.9999
Procedure      neurological examination      0.9999
Procedure                         power      0.9999
Procedure          physical examination      0.9998
  Symptom                         alert      0.9995
  Symptom                      suicidal      0.9994
  Symptom             abnormal movement      0.9994
Condition              cerebral infarct      0.9992
Condition                        stroke      0.9984
Procedure                         pulse      0.9972
Procedure                blood 

In [16]:
top10_df.to_csv("top10_entities_v2.csv", index=False)
deduped.to_csv("filtered_entities_v2.csv", index=False)
freq.to_csv("entity_frequencies_v2.csv", index=False)

print(f"Saved: top10_entities_v2.csv ({len(top10_df)} rows)")
print(f"Saved: filtered_entities_v2.csv ({len(deduped)} rows)")
print(f"Saved: entity_frequencies_v2.csv ({len(freq)} rows)")

Saved: top10_entities_v2.csv (40 rows)
Saved: filtered_entities_v2.csv (15934 rows)
Saved: entity_frequencies_v2.csv (6544 rows)
